[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/b_18_linear_regression_scan_solution.ipynb)

# 🟡 Solution: Linear Regression with lax.scan

*Training · Medium*

Reference implementation. Try it yourself in `b_18_linear_regression_scan.ipynb` first.

---
Problem 40 again, with the training loop rewritten as a `jax.lax.scan` —
and the loss curve falling out of it for free.

$$\hat{y} = Xw + b, \qquad \mathcal{L} = \frac{1}{N}\|Xw + b - y\|^2$$

### Signature
```python
class LinearRegressionScan:
    def closed_form(self, X, y): ...                          # -> (w, b)
    def gradient_descent(self, X, y, lr=0.01, steps=1000): ...# -> (w, b, losses)
    def nn_linear(self, X, y, lr=0.01, steps=1000): ...       # -> (w, b)
```

`X` is `(N, D)`, `y` is `(N,)`, `w` comes back `(D,)` and `b` is a scalar.

### What changes from problem 40
Only `gradient_descent`, and it changes in two ways:

1. **No Python `for` loop.** The update goes in a `lax.scan` body.
2. **It returns a third value**, `losses` of shape `(steps,)` — the MSE
   *before* each update, so `losses[0]` is the loss at the starting point
   `w = 0, b = 0`. This is not extra work: it is the `ys` that scan stacks
   for you, and it is the reason to reach for scan rather than
   `fori_loop`.

`closed_form` and `nn_linear` are exactly as in problem 40. Keep them —
`closed_form` is what the scan version has to agree with.

### Fitting a training loop into scan
Scan wants `(carry, x) -> (carry, y)`. A training loop has no per-step input,
so `xs` is `None` and the trip count comes from `length=`:

```python
def step(carry, _):
    w, b = carry
    ...
    return (w_new, b_new), loss        # (new carry, per-step output)

(w, b), losses = jax.lax.scan(step, (w0, b0), None, length=steps)
```

The gradients are still the ones you derive by hand — no `jax.grad`:

$$\nabla_w = \frac{2}{N}X^\top(\hat{y}-y), \qquad
  \nabla_b = \frac{2}{N}\sum(\hat{y}-y)$$

### Why this is the version that matters
A Python loop is unrolled at trace time: 2000 steps means a 2000-node graph,
and XLA has to compile every one of them. `scan` compiles the body **once**
and loops it, so compile time is flat in `steps`. Measured on this problem:

| | steps | wall clock |
|---|---|---|
| Python loop (problem 40) | 2 000 | 31.4 s |
| `lax.scan` | 20 000 | 0.04 s |

Ten times the steps, roughly a thousandth of the time. That gap is not an
optimisation detail — it is the difference between a training loop you can
`jit` and one you cannot.

`steps` has to be **static**, because it is the length of the scan and
therefore part of the shape of `losses`. Under `jit` that means
`static_argnames=('steps',)`.

### Where you have met this before
This is the same shape as `b_06`'s discounted returns — carry a state, emit
one value per step. An optimizer loop, an RNN, a KV-cache decode and a
diffusion sampler are all this one pattern; only the carry changes.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp
from flax import nnx


class LinearRegressionScan:
    def closed_form(self, X, y):
        N, D = X.shape
        X_aug = jnp.concatenate([X, jnp.ones((N, 1))], axis=1)   # (N, D+1)
        theta = jnp.linalg.lstsq(X_aug, y)[0]                    # (D+1,)
        return theta[:D], theta[D]

    def gradient_descent(self, X, y, lr=0.01, steps=1000):
        N, D = X.shape

        def step(carry, _):
            w, b = carry
            error = X @ w + b - y                # (N,)
            loss = jnp.mean(error ** 2)          # BEFORE the update
            grad_w = (2.0 / N) * (X.T @ error)   # (D,)
            grad_b = (2.0 / N) * jnp.sum(error)  # scalar
            return (w - lr * grad_w, b - lr * grad_b), loss

        # No per-step input, so xs=None and the trip count comes from length=.
        init = (jnp.zeros(D), jnp.array(0.0))
        (w, b), losses = jax.lax.scan(step, init, None, length=steps)
        return w, b, losses

    def nn_linear(self, X, y, lr=0.01, steps=1000):
        D = X.shape[1]
        layer = nnx.Linear(D, 1, rngs=nnx.Rngs(params=0))

        def loss_fn(model, X, y):
            return jnp.mean((model(X).squeeze(-1) - y) ** 2)

        grad_fn = nnx.grad(loss_fn)          # transform once, not per step

        for _ in range(steps):
            grads = grad_fn(layer, X, y)
            # One tree.map updates every parameter, whatever the module is.
            params = nnx.state(layer, nnx.Param)
            nnx.update(layer, jax.tree.map(lambda p, g: p - lr * g, params, grads))

        # Flax kernel is (D, 1) — the transpose of torch's (1, D).
        return layer.kernel[...].squeeze(-1), layer.bias[...].squeeze()

In [ ]:
# 🔍 Verify
import jax
import jax.numpy as jnp

X = jax.random.normal(jax.random.key(0), (200, 3))
w_true = jnp.array([2.0, -3.0, 0.5])
y = X @ w_true + 1.5

model = LinearRegressionScan()
w_cf, b_cf = model.closed_form(X, y)
w_gd, b_gd, losses = model.gradient_descent(X, y, lr=0.1, steps=2000)

print(f"closed_form       w={jnp.round(w_cf, 3)}  b={float(b_cf):.3f}")
print(f"gradient_descent  w={jnp.round(w_gd, 3)}  b={float(b_gd):.3f}")
print(f"truth             w={w_true}  b=1.5")

# The loss curve came free with the scan — no extra pass over the data.
print(f"\nlosses {losses.shape}: {float(losses[0]):.4f} -> {float(losses[-1]):.6f}")
for i in (0, 1, 2, 10, 100, 1999):
    print(f"  step {i:>4}: {float(losses[i]):.6f}")

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("linear_regression_scan")